# What could I know before tipoff?
I build each team's history before pairing it with its opponent. Current-game box scores are outcomes, never inputs.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd() if (Path.cwd() / 'Data').exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
TABLES = ROOT / 'artifacts/tables'
pd.set_option('display.max_columns', 12)

## Season strength uses earlier totals
Offensive rating is 100 times prior points divided by prior possessions. Counts are shifted before expanding or rolling; ratios weight games by their denominators instead of averaging noisy game ratings.

In [2]:
from src.data import load_data, validate_data
from src.features import build_team_history, build_roster_history
raw_team, raw_player = load_data()
games, _, _, _ = validate_data(raw_team, raw_player)
history = build_team_history(games)
okc = history[(history.season == 2018) & (history.off_team == 'OKC')].copy()
display(okc[['gamedate','prior_games','history_end','net_rating','last10_net_rating','rest_days']].head(12))

,gamedate,prior_games,history_end,net_rating,last10_net_rating,rest_days
11480,2018-10-16,0,NaT,NaN,NaN,NaN
11481,2018-10-19,1,2018-10-16,-9.843314,-9.843314,2.0
11482,2018-10-21,2,2018-10-19,-12.881406,-12.881406,1.0
11483,2018-10-25,3,2018-10-21,-11.254019,-11.254019,3.0
11484,2018-10-28,4,2018-10-25,-10.481885,-10.481885,2.0
11485,2018-10-30,5,2018-10-28,-6.952777,-6.952777,1.0
11486,2018-11-01,6,2018-10-30,-3.041376,-3.041376,1.0
11487,2018-11-02,7,2018-11-01,-2.071614,-2.071614,0.0
11488,2018-11-05,8,2018-11-02,0.934106,0.934106,2.0
11489,2018-11-07,9,2018-11-05,1.464832,1.464832,1.0


The first game has no season history. Prediction coverage starts when both teams have ten earlier games; opening games still supply history and appear in the explorers. Rest uses known game dates, with days off capped at seven so unusually long breaks cannot dominate a linear coefficient.

## A thirty-minute player should count more than a four-minute player
For players on the last observed roster, normal role is minutes per appearance over the prior twenty team games. Normalized roles weight prior-five-game scoring efficiency, participation and absence history. Players without recent shot attempts are excluded from scoring efficiency and the remaining weights are renormalized.

In [3]:
roster, roles = build_roster_history(okc, raw_player)
game_id = int(okc.iloc[35].nbagameid)
example = roles[roles.nbagameid.eq(game_id)].sort_values('role_weight',ascending=False)
display(example[['player_name','role_minutes','role_weight','prior5_participation','prior5_ppa']].round(3))
print('Role weights sum to:',example.role_weight.sum())
display(roster[roster.nbagameid.eq(game_id)])

,player_name,role_minutes,role_weight,prior5_participation,prior5_ppa
480,Russell Westbrook,35.611,0.129,1.0,0.853
481,Paul George,35.360,0.128,0.8,1.218
485,Steven Adams,33.800,0.122,1.0,1.231
487,Jerami Grant,31.889,0.115,1.0,1.179
484,Dennis Schröder,29.170,0.105,1.0,0.933
490,Terrance Ferguson,18.608,0.067,0.8,1.067
486,Alex Abrines,18.008,0.065,0.4,1.100
483,Nerlens Noel,13.923,0.050,1.0,1.000
482,Patrick Patterson,12.985,0.047,1.0,1.375
491,Hamidou Diallo,11.495,0.042,1.0,1.300


Role weights sum to: 0.9999999999999999


,nbagameid,off_team,player_history_end,top5_share,top7_share,rotation_size,...,rotation_volatility,scoring_concentration,usage_concentration,weighted_ppa,weighted_absence,weighted_participation
35,21800547,OKC,2018-12-30,0.689886,0.811901,8.6,...,0.152671,0.150851,0.14861,1.11109,0.095115,0.822077


A `missed` flag does not establish that information was public before tipoff. Only earlier flags are used. Current roster membership is also excluded: the previous roster is a conservative proxy, with unavoidable lag around arrivals and departures.

## What does a matchup row look like?

In [4]:
matchups = pd.read_csv(TABLES / 'matchups.csv.gz',parse_dates=['gamedate','home_history_end','away_history_end','home_player_history_end','away_player_history_end'])
cols=['gamedate','home_off_team','away_off_team','net_rating_diff','win_pct_diff','weighted_ppa_diff','weighted_absence_diff','y']
display(matchups.loc[matchups.nbagameid.eq(game_id),cols])
for side in ['home','away']:
    for source in ['history_end','player_history_end']:
        end=matchups[f'{side}_{source}']
        assert (end.isna() | (end < matchups.gamedate)).all()
print('Every recorded history ends before its game date.')
display(pd.read_csv(TABLES / 'coverage.csv'))

,gamedate,home_off_team,away_off_team,net_rating_diff,win_pct_diff,weighted_ppa_diff,weighted_absence_diff,y
5466,2018-12-31,OKC,DAL,5.339343,0.142857,-0.05046,-0.014344,1


Every recorded history ends before its game date.


,season,games,eligible,start,end
0,2014,1230,1072,2014-10-28,2015-04-15
1,2015,1230,1073,2015-10-27,2016-04-13
2,2016,1230,1074,2016-10-25,2017-04-12
3,2017,1230,1075,2017-10-17,2018-04-11
4,2018,1230,1074,2018-10-16,2019-04-10
5,2019,1059,906,2019-10-22,2020-08-14
6,2020,1080,924,2020-12-22,2021-05-16
7,2021,1230,1073,2021-10-19,2022-04-10
8,2022,1230,1073,2022-10-18,2023-04-09
9,2023,1230,1074,2023-10-24,2024-04-14


Inputs are home minus away; designated home court is represented by the intercept. I avoid putting offensive rating, defensive rating and their difference in the same model. Net rating supplies that summary.

## Can future records sneak backward?
The tests mutate current and future points, player minutes, starter flags and missed flags, then reconstruct histories. Earlier and same-day inputs must remain identical. Another test deletes future rows, including future players. The role-weighting test removes the entire current roster and checks that its pregame snapshot is unchanged. Run `python -m pytest tests/test_features.py -q`.